# 2.0 Knowledge Distillation for a Vision-Language Model

In this notebook you will learn how to:

- Apply text-only knowledge distillation to a pruned VLM, using the unpruned VLM as the teacher.
- Use [`distill.py`](../distill.py) from the Megatron-Bridge example, which auto-detects VLM inputs and handles the extract/reinsert plumbing for you.
- Confirm the distilled VLM still captions images correctly after the distilled language tower is spliced back in.

Prerequisites: notebook [`01_pruning_vlm.ipynb`](01_pruning_vlm.ipynb) has produced a pruned VLM at `/workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752`.

---
## 2.1 Background: text-only KD on the language tower

Knowledge distillation ([Hinton et al. 2015](https://arxiv.org/abs/1503.02531)) trains a smaller **student** to mimic the soft outputs of a larger **teacher** on the same inputs. For VLMs we apply the same idea but only to the language tower — for three reasons:

- The pruning step in notebook 01 only shrank the language tower; the vision encoder and projector are unchanged. So they don't need to learn anything.
- Text-only KD is much cheaper than multimodal KD (no visual token preprocessing, no image batches).
- The Minitron paper has shown that pure-text KD recovers most of the quality lost to structured pruning, even for downstream multimodal tasks, because the projector → language-tower interface is preserved.

Concretely, [`distill.py`](../distill.py) does this for VLM inputs:

1. Detects that `--student_hf_path` (and/or `--teacher_hf_path`) is a VLM.
2. Extracts each language tower to a temporary plain HF causal LM (rank 0 only, with barriers).
3. Loads both into Megatron-Core via `AutoBridge` and runs the standard `kd_loss` distillation loop on text tokens.
4. Exports the distilled student LM in HF format.
5. Reinserts that LM back into the original student VLM container, preserving the vision encoder, projector and `lm_head` bit-for-bit.

All five steps happen behind a single `torchrun` invocation.

---
## 2.2 (Optional) Watch progress in TensorBoard

The script writes TensorBoard logs to `<output_dir>/tb_logs`. Open them in another window before kicking off the run if you want to see the KL-divergence + CE loss curves live.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir /workspace/user_homes/lmikaelyan/Qwen3-VL-distill/tb_logs

---
## 2.3 Run distillation

Quickest possible smoke run: `--use_mock_data` skips dataset preparation entirely, the language tower trains for `--train_iters` steps on synthetic text, and `--hf_export_path` + `--student_hf_model` triggers the inline HF conversion plus VLM reinsertion at the end.

For a real run you'd swap `--use_mock_data` for `--data_paths 1.0 /path/to/tokenized_text_document`, raise `--train_iters` to a few thousand, and increase `--tp_size` / `--nproc_per_node` to whatever fits.

Note: this is the **VLM** flow. `--student_hf_path` and `--teacher_hf_path` both point at VLM checkpoints; the script handles the LM extraction internally.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 torchrun --nproc_per_node 1 ../distill.py \
    --student_hf_path /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752 \
    --teacher_hf_path /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct \
    --use_mock_data \
    --seq_length 512 --mbs 1 --gbs 1 \
    --train_iters 20 --lr 1e-5 --lr_warmup_iters 2 \
    --eval_interval 100 --eval_iters 1 --log_interval 5 \
    --output_dir /workspace/user_homes/lmikaelyan/Qwen3-VL-distill-smoke \
    --hf_export_path /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke \
    --student_hf_model /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752 \
    --trust_remote_code

Expected log markers:

```
Detected VLM student (...); extracting language tower to ..._vlm_student_lm.
Detected VLM teacher (...); extracting language tower to ..._vlm_teacher_lm.
Starting distillation...
Distillation done! Saved checkpoint to .../checkpoints in megatron distributed checkpoint format.
Exporting final distilled ckpt to HF format to ..._vlm_distilled_lm
Reinserting distilled LM from ..._vlm_distilled_lm into VLM container; writing final VLM to /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke.
```

### Real-data distillation

For a recovery-grade run, pre-tokenize a text corpus into Megatron's `.bin` / `.idx` format (see [`../dataset/README.md`](../../dataset/README.md)) and replace `--use_mock_data` with `--data_paths`:

```bash
torchrun --nproc_per_node 8 ../distill.py \
    --tp_size 8 \
    --student_hf_path /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752 \
    --teacher_hf_path /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct \
    --data_paths 1.0 /path/to/tokenized_qwen3/data_text_document \
    --seq_length 8192 --mbs 1 --gbs 768 \
    --train_iters 2000 --lr 1e-4 --min_lr 1e-5 --lr_warmup_iters 50 \
    --output_dir /workspace/user_homes/lmikaelyan/Qwen3-VL-distill \
    --hf_export_path /workspace/user_homes/lmikaelyan/Qwen3-VL-distilled \
    --student_hf_model /workspace/user_homes/lmikaelyan/Qwen3-VL-8B-Instruct-pruned-34-32-10752 \
    --trust_remote_code
```

---
## 2.4 Verify the distilled VLM is still multimodal

Distillation only touched the language tower. Load the reassembled VLM and confirm it still captions an image — the vision encoder and projector were spliced back in unchanged, so multimodal alignment should be intact.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import requests, torch

DISTILLED_PATH = "/workspace/user_homes/lmikaelyan/Qwen3-VL-distilled-smoke"
vlm = AutoModelForImageTextToText.from_pretrained(
    DISTILLED_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(DISTILLED_PATH, trust_remote_code=True)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "Describe the image in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
).to(vlm.device)
out = vlm.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0])

In [ ]:
import gc, torch
del vlm; gc.collect(); torch.cuda.empty_cache()

A 20-iter mock-data run won't actually recover quality — it's a structural smoke test of the extract → distill → reinsert pipeline. For real recovery, use the larger-data command in §2.3 and run for a few thousand steps on a real text corpus.